# GAN with LSUN Dataset (Bedroom Scenes)

## **Install Dependencies**

LSUN is loaded via TensorFlow Datasets (`tfds`). We install it first.

We also need enough GPU memory — make sure you are using a **GPU runtime** in Colab (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip install tensorflow-datasets -q

## **Setup and Data Preprocessing**

We prepare the LSUN Bedroom dataset.

Key differences from MNIST:
- Images are **RGB (3 channels)** instead of grayscale (1 channel)
- We resize all images to **64×64** pixels for a manageable resolution
- GANs are very sensitive to data scaling, so we normalize images to **[-1, 1]** to match the `tanh` output of the Generator

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
import numpy as np

# Image size and channels — LSUN is color (RGB), resized to 64x64
IMG_SIZE = 64
CHANNELS = 3  # RGB (changed from 1 in MNIST)

# Set constants for batching
BUFFER_SIZE = 10000
BATCH_SIZE = 128  # Reduced from 256 because 64x64 RGB images use more GPU memory

def preprocess(example):
    """
    Resize each image to 64x64, cast to float32,
    and normalize pixels from [0, 255] to [-1, 1]
    """
    image = example['image']
    # Resize to 64x64
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # Cast to float32
    image = tf.cast(image, tf.float32)
    # Normalize to [-1, 1] to match generator's tanh output
    image = (image - 127.5) / 127.5
    return image

# Load the LSUN bedroom split from TensorFlow Datasets
# The first run will download the dataset — this may take a while
dataset, info = tfds.load('lsun/bedroom', split='train', with_info=True)

# Apply preprocessing, shuffle, and batch
train_dataset = (
    dataset
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)  # Prefetch for faster GPU pipeline
)

print(f"Dataset loaded. Image shape: {IMG_SIZE}x{IMG_SIZE}x{CHANNELS}")

## **Creating the Generator**

The Generator's job is to take a "seed" (random noise) and turn it into a **64×64 RGB image**.

Key differences from the MNIST Generator:
- We start from a **4×4** base (instead of 7×7) and upsample 4 times to reach 64×64
- The final layer outputs **3 channels** (RGB) instead of 1
- We use more filters (512 at the start) to capture richer scene features

Upsampling path: `4×4 → 8×8 → 16×16 → 32×32 → 64×64`

In [ ]:
def make_generator_model():
    model = tf.keras.Sequential([
        # Take 100 random numbers and project into 4x4x512 nodes
        # (Larger base than MNIST's 7x7x256 because we need to reach 64x64)
        layers.Dense(4 * 4 * 512, use_bias=False, input_shape=(100,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Reshape flat vector into a 4x4 spatial feature map with 512 channels
        layers.Reshape((4, 4, 512)),

        # Upsample: 4x4 -> 8x8
        layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Upsample: 8x8 -> 16x16
        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Upsample: 16x16 -> 32x32
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Upsample: 32x32 -> 64x64 (Final Image Size)
        # Output 3 channels for RGB (changed from 1 in MNIST)
        # Tanh activation ensures output is in [-1, 1] range
        layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')
    ])
    return model

# Instantiate the generator
generator = make_generator_model()
generator.summary()

## **Creating the Discriminator**

The Discriminator is essentially an image classifier that tries to distinguish "Real" from "Fake."

Key differences from the MNIST Discriminator:
- Input shape is **64×64×3** (RGB) instead of 28×28×1
- We add an **extra Conv2D block** to progressively downsample the larger 64×64 image
- More filters (128 → 256) to handle the complexity of real-world scenes

Downsampling path: `64×64 → 32×32 → 16×16 → 8×8 → Flatten → Score`

In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential([
        # Downsample: 64x64 -> 32x32
        # Input is RGB so input_shape has 3 channels (changed from 1 in MNIST)
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[IMG_SIZE, IMG_SIZE, CHANNELS]),
        layers.LeakyReLU(),
        # Dropout prevents discriminator from becoming too strong too fast
        layers.Dropout(0.3),

        # Downsample: 32x32 -> 16x16
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        # Extra block needed for 64x64 input (not present in MNIST version)
        # Downsample: 16x16 -> 8x8
        layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        # Flatten 3D features into a 1D vector
        layers.Flatten(),
        # Output a single score (how real the image looks)
        layers.Dense(1)
    ])
    return model

# Instantiate the discriminator
discriminator = make_discriminator_model()
discriminator.summary()

## **Defining Loss and Optimizers**

The loss functions define the mathematical "competition" between the two models.

This section is **identical to the MNIST version** — the GAN loss logic does not change with the dataset.

In [ ]:
# Binary Cross Entropy measures how far the prediction is from the target (0 or 1)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    # Discriminator wants real images to be 1s
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    # Discriminator wants fake images to be 0s
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    # Generator wants the discriminator to believe the fakes are 1s (real)
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Using Adam optimizer with a slightly lower learning rate (2e-4)
# LSUN is more complex than MNIST, so a conservative LR helps training stability
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

## **The Training Step**

This is the heart of the GAN. We use `tf.GradientTape` to track calculations and compute gradients for backpropagation.

This is **identical to the MNIST version** in structure — the same generator/discriminator competition logic applies regardless of dataset.

In [ ]:
@tf.function  # This annotation compiles the function for faster GPU execution
def train_step(images):
    # Create random noise as input for the generator
    noise = tf.random.normal([BATCH_SIZE, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # 1. Generator creates 64x64 RGB images from noise
        generated_images = generator(noise, training=True)

        # 2. Discriminator evaluates real and fake images
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        # 3. Calculate losses
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    # 4. Calculate gradients (direction to move weights to reduce loss)
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # 5. Apply gradients to update model weights
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

# Loop through the dataset for a specific number of epochs
def train(dataset, epochs):
    for epoch in range(epochs):
        gen_loss_avg = []
        disc_loss_avg = []

        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch)
            gen_loss_avg.append(g_loss)
            disc_loss_avg.append(d_loss)

        # Print average losses per epoch so you can monitor training health
        print(f"Epoch {epoch+1} complete | "
              f"Gen Loss: {np.mean(gen_loss_avg):.4f} | "
              f"Disc Loss: {np.mean(disc_loss_avg):.4f}")

# Call this to start training
# LSUN needs more epochs than MNIST due to scene complexity
# 30-50 epochs is a good starting point with a GPU runtime
train(train_dataset, 50)

## **Visualize Generated Images**

After training, generate some sample bedroom images and display them.

This bonus section was not in the original MNIST notebook but is useful to visually inspect GAN output quality.

In [ ]:
import matplotlib.pyplot as plt

def generate_and_plot(generator, num_images=16):
    # Generate random noise vectors
    noise = tf.random.normal([num_images, 100])
    # Use the trained generator to produce images
    generated_images = generator(noise, training=False)
    # Denormalize from [-1, 1] back to [0, 1] for display
    generated_images = (generated_images + 1) / 2.0

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(generated_images[i].numpy())
        ax.axis('off')
    plt.suptitle('Generated LSUN Bedroom Scenes', fontsize=16)
    plt.tight_layout()
    plt.show()

generate_and_plot(generator)